<a href="https://colab.research.google.com/github/uditmodi2322-bot/Gold-predictions/blob/main/micro_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os

# 1. Download the latest version
# This defines the variable 'path'
path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
print("Path to dataset files:", path)

# 2. List files in the path to find the CSV name
# Now 'path' is definitely defined!
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
data_path = os.path.join(path, csv_file)

# 3. Load the dataset
df = pd.read_csv(data_path)
print("Successfully loaded the dataframe!")
print(df.head())

num_rows = len(df)
print(f"The dataset contains {num_rows} rows of crop data.")

100%|██████████| 63.7k/63.7k [00:00<00:00, 510kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/atharvaingle/crop-recommendation-dataset/versions/1
Successfully loaded the dataframe!
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice
The dataset contains 2200 rows of crop data.


In [ ]:
from sklearn.model_selection import train_test_split


# Features (N, P, K, temperature, humidity, ph, rainfall)
X = df.drop('label', axis=1)

# Target (the crop name)
y = df['label']

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import pandas as pd

# 1. Initialize the Model
# n_estimators=100 means we are using 100 decision trees for a 'majority vote'
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the Model
# Ensure X_train is a DataFrame so the model remembers the feature names (N, P, K, etc.)
model.fit(X_train, y_train)

# 3. Evaluate the Model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) * 100
print(f"✅ Model Accuracy: {accuracy:.2f}%")

# 4. SAVE THE MODEL
# This 'crop_model.pkl' is what your Django 'joblib.load' is looking for
joblib.dump(model, 'crop_model.pkl')
print("📂 SUCCESS: 'crop_model.pkl' created. Refresh your Colab sidebar to see it.")

# 5. Visualizing the Confusion Matrix (Uncomment to see performance)
#plt.figure(figsize=(12, 8))
#cm = confusion_matrix(y_test, y_pred)
#sns.heatmap(cm, annot=False, cmap='YlGnBu')
#plt.title('Crop Classification Performance')
#plt.xlabel('Predicted Crops')
#plt.ylabel('Actual Crops')
#plt.show()

✅ Model Accuracy: 100.00%
📂 SUCCESS: 'crop_model.pkl' created. Refresh your Colab sidebar to see it.


In [ ]:
import pandas as pd

print("--- Enter Soil and Environmental Data ---")

# 1. Collecting all 7 inputs (The "Subsets" of your data)
n = float(input("Enter Nitrogen (N): "))
p = float(input("Enter Phosphorus (P): "))
k = float(input("Enter Potassium (K): "))
temp = float(input("Enter Temperature (°C): "))
hum = float(input("Enter Humidity (%): "))
ph = float(input("Enter Soil pH: "))
rain = float(input("Enter Rainfall (mm): "))

# 2. Organize into a DataFrame to match the Training Data structure
# Note: The column names MUST match exactly what you used during model.fit()
user_data = pd.DataFrame([[n, p, k, temp, hum, ph, rain]],
                         columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])

# 3. Predict using your Supervised Learning Model
# (Assuming your model is named 'model')
prediction = model.predict(user_data)

print("\n-------------------------------------------")
print(f"RESULT: The best crop to plant is **{prediction[0].upper()}**")
print("-------------------------------------------")

--- Enter Soil and Environmental Data ---
Enter Nitrogen (N): 89
Enter Phosphorus (P): 80
Enter Potassium (K): 10
Enter Temperature (°C): 23.5
Enter Humidity (%): 50
Enter Soil pH: 6.5
Enter Rainfall (mm): 75

-------------------------------------------
RESULT: The best crop to plant is **MAIZE**
-------------------------------------------


In [ ]:
!pip install django

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 57.3 MB/s eta 0:00:00


In [ ]:
!pip install django pyngrok

In [ ]:
import os

# Create the directory
os.makedirs('templates', exist_ok=True)

# Write the HTML file
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>NexusAI - Crop Recommendation</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="bg-light">
    <div class="container mt-5">
        <div class="card shadow p-4">
            <h2 class="text-center text-success">🌱 NexusAI Crop Predictor</h2>
            <hr>
            <form method="POST">
                <div class="row mt-3">
                    <div class="col-md-4">Nitrogen (N): <input type="number" name="n" class="form-control" required></div>
                    <div class="col-md-4">Phosphorus (P): <input type="number" name="p" class="form-control" required></div>
                    <div class="col-md-4">Potassium (K): <input type="number" name="k" class="form-control" required></div>
                </div>
                <div class="row mt-3">
                    <div class="col-md-6">Temperature (°C): <input type="number" step="0.01" name="temp" class="form-control" required></div>
                    <div class="col-md-6">Humidity (%): <input type="number" step="0.01" name="hum" class="form-control" required></div>
                </div>
                <div class="row mt-3">
                    <div class="col-md-6">pH Level: <input type="number" step="0.01" name="ph" class="form-control" required></div>
                    <div class="col-md-6">Rainfall (mm): <input type="number" step="0.01" name="rain" class="form-control" required></div>
                </div>
                <button type="submit" class="btn btn-success w-100 mt-4">Predict Best Crop</button>
            </form>

            {% if crop %}
            <div class="alert alert-info mt-4 text-center">
                <h3>Recommended Crop: <strong>{{ crop }}</strong></h3>
            </div>
            {% endif %}
        </div>
    </div>
</body>
</html>
"""

with open('templates/predict.html', 'w') as f:
    f.write(html_content)

In [ ]:
import os
import sys
import threading
import time
import joblib
import pandas as pd
from django.conf import settings
from django.core.management import execute_from_command_line
from django.shortcuts import render
from django.urls import path

# 1. DJANGO SETTINGS
if not settings.configured:
    settings.configure(
        DEBUG=True,
        ALLOWED_HOSTS=['*'],
        ROOT_URLCONF=__name__,
        SECRET_KEY='nexusai-key-123',
        TEMPLATES=[{
            'BACKEND': 'django.template.backends.django.DjangoTemplates',
            'DIRS': [os.path.join(os.getcwd(), 'templates')],
        }],
    )

# 2. THE LOGIC (View)
def home_view(request):
    result = None
    if request.method == 'POST':
        try:
            # Load your saved brain (the model)
            model = joblib.load('crop_model.pkl')

            # Collect data from form
            data = [
                float(request.POST.get('n')), float(request.POST.get('p')), float(request.POST.get('k')),
                float(request.POST.get('temp')), float(request.POST.get('hum')),
                float(request.POST.get('ph')), float(request.POST.get('rain'))
            ]

            # Predict
            df_input = pd.DataFrame([data], columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])
            result = model.predict(df_input)[0]
        except Exception as e:
            result = f"Error: {e}"

    return render(request, 'predict.html', {'crop': result})

# 3. URLS & SERVER THREAD
urlpatterns = [path('', home_view)]

def run_django():
    sys.argv = ['manage.py', 'runserver', '0.0.0.0:8000', '--noreload']
    execute_from_command_line(sys.argv)

# Start server in background
threading.Thread(target=run_django, daemon=True).start()
print("✅ Django Server is warming up on Port 8000...")

✅ Django Server is warming up on Port 8000...


In [ ]:
# Install Localtunnel
!npm install -g localtunnel

# Start the tunnel
import time
time.sleep(2)
print("\n--- INSTRUCTIONS ---")
print("1. Click the link below.")
print("2. If it asks for a 'Tunnel Password', copy-paste this number: ")
!curl https://loca.lt/mytunnelpassword
print("---------------------\n")

!npx localtunnel --port 8000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼
--- INSTRUCTIONS ---
1. Click the link below.
2. If it asks for a 'Tunnel Password', copy-paste this number: 
104.199.138.201---------------------

⠙⠹⠸⠼⠴your url is: https://every-ravens-rhyme.loca.lt


[08/Apr/2026 08:51:19] "GET / HTTP/1.1" 200 1707
Not Found: /favicon.ico
[08/Apr/2026 08:51:19] "GET /favicon.ico HTTP/1.1" 404 2211
[08/Apr/2026 08:52:46] "GET / HTTP/1.1" 200 1707
[08/Apr/2026 08:53:02] "GET / HTTP/1.1" 200 1707
[08/Apr/2026 08:53:24] "GET / HTTP/1.1" 200 1707
[08/Apr/2026 08:53:35] "GET / HTTP/1.1" 200 1707
Not Found: /favicon.ico
[08/Apr/2026 08:53:36] "GET /favicon.ico HTTP/1.1" 404 2211
[08/Apr/2026 08:54:11] "POST / HTTP/1.1" 200 1867
[08/Apr/2026 08:54:54] "POST / HTTP/1.1" 200 1871
[08/Apr/2026 08:56:38] "GET / HTTP/1.1" 200 1707


^C
